# NHL win-probability model

Hockey has no ties (OT/shootout always decides), so it's a **binary**, goal-margin sport — the *same* sport-agnostic `walk_forward` Elo+feature pipeline as NBA/MLB, on **18k real NHL games** (2010–2026, `data/nhl.duckdb`, free NHL web API). Out-of-sample.

Expect baseball-like modest skill: hockey is high-variance and the dominant per-game lever — the **starting goalie** — is invisible to team-Elo.

In [ ]:
import sys
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import os as _os
if _os.getenv('NB_DARK'):
    sns.set_theme(style='darkgrid', rc={
        'figure.facecolor':'#0d1117','axes.facecolor':'#161b22','savefig.facecolor':'#0d1117',
        'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9','text.color':'#c9d1d9',
        'axes.titlecolor':'#c9d1d9','xtick.color':'#8b949e','ytick.color':'#8b949e',
        'grid.color':'#21262d'})
else:
    sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'src'/'sportsball').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src')); sys.path.insert(0, str(ROOT/'scripts'))
from sportsball.pipelines._elo import walk_forward
from train_eval_duckdb import _pipeline
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score
K, HFA = 6, 30

In [ ]:
con = duckdb.connect(str(ROOT/'data'/'nhl.duckdb'), read_only=True)
raw = con.execute('SELECT game_date,home_team,away_team,home_score,away_score FROM games ORDER BY game_date, game_id').fetchall()
con.close()
results = [(d,h,a,hs,as_) for (d,h,a,hs,as_) in raw]
frows, snaps = walk_forward(results, K, HFA, mov_enabled=True, carry=0.75, gap_days=120, form_window=15)
X = np.array([r.features for r in frows]); y = np.array([1 if r.actual>=1 else 0 for r in frows])
elo_p = np.array([r.exp_home for r in frows])
cut = int(0.85*len(X)); base = y[cut:].mean()
print(f'{len(X)} games | test {len(y)-cut} | home-win base rate {base:.3f}')

## How much skill? (out-of-sample holdout)

In [ ]:
def report(name, p):
    return {'model':name,'accuracy':round(accuracy_score(y[cut:],(p[cut:]>=0.5).astype(int)),4),
            'log_loss':round(log_loss(y[cut:],p[cut:],labels=[0,1]),4),'brier':round(brier_score_loss(y[cut:],p[cut:]),4)}
home_const = np.full(len(y), y[:cut].mean())
logit=_pipeline().fit(X[:cut],y[:cut]); full_p=np.empty(len(y)); full_p[:cut]=logit.predict_proba(X[:cut])[:,1]; full_p[cut:]=logit.predict_proba(X[cut:])[:,1]
pd.DataFrame([report('always-home (base rate)', home_const), report('Elo only', elo_p), report('full-feature logistic', full_p)])

## Calibration

In [ ]:
from sklearn.calibration import calibration_curve
fp, mp = calibration_curve(y[cut:], full_p[cut:], n_bins=8, strategy='quantile')
plt.figure(figsize=(6,6)); plt.plot([0,1],[0,1],'--',color='gray',label='perfect')
plt.plot(mp, fp, 'o-', color='steelblue', label='NHL model'); plt.xlabel('predicted P(home win)'); plt.ylabel('observed')
plt.title('NHL model calibration (holdout)'); plt.legend(); plt.show()

## Team strength (final Elo) & skill over time

In [ ]:
rate = pd.DataFrame([(t,s.elo) for t,s in snaps.items()], columns=['team','elo']).sort_values('elo', ascending=False)
fig, ax = plt.subplots(1,2, figsize=(13,5))
sns.barplot(data=rate.head(8), y='team', x='elo', ax=ax[0], color='steelblue'); ax[0].set_title('Top 8 by Elo')
sns.barplot(data=rate.tail(8), y='team', x='elo', ax=ax[1], color='indianred'); ax[1].set_title('Bottom 8 by Elo')
for a in ax: a.set_xlim(rate.elo.min()-20, rate.elo.max()+20)
plt.tight_layout(); plt.show()
win=400; correct=(full_p[cut:]>=0.5).astype(int)==y[cut:]
plt.figure(figsize=(11,4))
plt.plot(pd.Series(correct).rolling(win,min_periods=50).mean().values, color='steelblue', label='model')
plt.plot(pd.Series(y[cut:]==1).rolling(win,min_periods=50).mean().values, color='gray', ls='--', label='always-home')
plt.axhline(0.5,color='crimson',lw=.6); plt.legend(); plt.ylabel(f'rolling accuracy ({win})'); plt.title('NHL skill over time'); plt.show()

## Verdict

Real but modest skill — a few points above always-picking-home, log-loss below a coin flip, sensible Elo ratings. Like baseball, the ceiling is a single dominant per-game factor team-Elo can't see: the **starting goalie** (the NHL analog of the MLB starting pitcher). A goalie rating (save% / goals-saved-above-expected, point-in-time) is the obvious next feature; rest/back-to-back already help (hockey plays dense schedules). Refresh with `research/nhl/ingest_nhl.py`.